<a href="https://colab.research.google.com/github/CassieMarie0728/colab-notebooks/blob/main/JSON_to_CSV_Converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JSON to CSV Converter - The No-Nonsense Edition

## What This Badass Notebook Does

Listen, you've got a ton of JSON files and you're tired of manually combining them like some kind of medieval peasant. This notebook is your chainsaw. It:

1. **Uploads multiple JSON files** - Yeah, like, **a LOT** of them. Throw 100+ at it if you want.
2. **Parses each one** - Handles nested structures like a boss.
3. **Flattens the data** - Turns those nested monstrosities into flat, normalizable records.
4. **Combines everything** - Merges all the data into one coherent dataframe.
5. **Exports to CSV** - Downloads it to your machine faster than you can say 'data pipeline'.

**No external dependencies you don't already have. No bloated libraries. Just pure, unadulterated efficiency.**

## Step 1: Install & Import Everything You Need

Yeah, I know, imports are boring as hell. But we need these to make the magic happen.

In [ ]:
import json
import pandas as pd
import numpy as np
from google.colab import files
from pathlib import Path
import io
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports loaded. We're ready to rock.")
print(f"✓ Pandas version: {pd.__version__}")
print("✓ Let's get this show on the road.")

## Step 2: Upload Your JSON Files

Time to throw your data at the notebook. Don't be shy—upload as many as you want. The notebook can handle it.

In [ ]:
print("🔥 Ready to upload? Click the 'Choose Files' button and select ALL your JSON files.")
print("💪 Select multiple files at once—you know you can, right?\n")

uploaded_files = files.upload()

print(f"\n✓ Uploaded {len(uploaded_files)} file(s).")
for filename in uploaded_files.keys():
    print(f"  - {filename}")

## Step 3: Define Helper Functions

These bad boys do the heavy lifting. They flatten nested structures, handle edge cases, and make sure your data doesn't look like a Lovecraftian nightmare.

In [ ]:
def flatten_json(y, parent_key='', sep='_'):
    """
    Flatten nested JSON structures into a single-level dictionary.

    This function recursively walks through nested dictionaries and lists,
    creating flattened keys like 'parent_child_grandchild'.

    Args:
        y (dict): The JSON object to flatten
        parent_key (str): The parent key for recursive calls
        sep (str): The separator between keys

    Returns:
        dict: Flattened dictionary
    """
    items = []

    if isinstance(y, dict):
        for k, v in y.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k

            if isinstance(v, dict):
                items.extend(flatten_json(v, new_key, sep=sep).items())
            elif isinstance(v, list):
                # Handle lists: either flatten them or convert to string
                if len(v) == 0:
                    items.append((new_key, None))
                elif isinstance(v[0], dict):
                    # For list of dicts, convert to JSON string
                    items.append((new_key, json.dumps(v)))
                else:
                    # For simple lists, join as string
                    items.append((new_key, '|'.join(map(str, v))))
            else:
                items.append((new_key, v))

    return dict(items)


def process_json_file(file_content, filename):
    """
    Process a single JSON file and return a list of flattened records.

    Handles:
    - Single JSON objects
    - JSON arrays
    - Malformed JSON (returns None)

    Args:
        file_content (bytes): The raw file content
        filename (str): The name of the file (for logging)

    Returns:
        list: List of flattened dictionaries, or None if error
    """
    try:
        data = json.loads(file_content)

        # Handle both single objects and arrays
        if isinstance(data, list):
            records = [flatten_json(item) for item in data]
        elif isinstance(data, dict):
            records = [flatten_json(data)]
        else:
            print(f"⚠️  {filename}: JSON is neither array nor object. Skipping.")
            return None

        return records

    except json.JSONDecodeError as e:
        print(f"❌ {filename}: Invalid JSON. Error: {e}")
        return None
    except Exception as e:
        print(f"❌ {filename}: Unexpected error. {type(e).__name__}: {e}")
        return None


def standardize_columns(df):
    """
    Clean up dataframe columns: lowercase, remove spaces, standardize names.

    Args:
        df (pd.DataFrame): The dataframe to standardize

    Returns:
        pd.DataFrame: Dataframe with cleaned column names
    """
    df.columns = (df.columns
                  .str.lower()
                  .str.replace(' ', '_')
                  .str.replace('[^a-z0-9_]', '', regex=True))
    return df


print("✓ Helper functions defined and ready to destroy.")

## Step 4: Process All JSON Files

Here's where the magic happens. We're gonna loop through every single file you uploaded and extract the data like we're mining for gold.

In [ ]:
# Storage for all records
all_records = []
files_processed = 0
files_failed = 0
total_records = 0

print("🚀 Processing files...\n")
print("-" * 60)

for filename, file_data in uploaded_files.items():
    # Make sure it's a JSON file
    if not filename.lower().endswith('.json'):
        print(f"⊘  {filename}: Not a JSON file. Skipping.")
        continue

    # Decode the file content
    try:
        file_content = file_data.decode('utf-8')
    except UnicodeDecodeError:
        print(f"❌ {filename}: Encoding error. Not UTF-8 encoded.")
        files_failed += 1
        continue

    # Process the file
    records = process_json_file(file_content, filename)

    if records is not None:
        all_records.extend(records)
        record_count = len(records)
        total_records += record_count
        files_processed += 1
        print(f"✓ {filename:<40} | {record_count:>5} records extracted")
    else:
        files_failed += 1

print("-" * 60)
print(f"\n📊 Processing Summary:")
print(f"   Files processed: {files_processed}")
print(f"   Files failed: {files_failed}")
print(f"   Total records extracted: {total_records}")

if all_records:
    print(f"\n✓ Ready to convert to CSV.")
else:
    print(f"\n❌ No valid records found. Check your JSON files, buddy.")

## Step 5: Convert to DataFrame

Combine all those records into a single, beautiful dataframe. This is where things get organized.

In [ ]:
if all_records:
    # Create dataframe from all records
    df = pd.DataFrame(all_records)

    print(f"📋 Dataframe created:")
    print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\n🔍 First few rows:")
    print(df.head())
    print(f"\n📌 Column names ({len(df.columns)} total):")
    for i, col in enumerate(df.columns, 1):
        print(f"   {i:>2}. {col}")

    print(f"\n💾 Data types:")
    print(df.dtypes)
    print(f"\n❄️  Missing values:")
    missing = df.isnull().sum()
    if missing.any():
        print(missing[missing > 0])
    else:
        print("   None! Your data is pristine.")
else:
    print("❌ Can't create dataframe—no records to work with.")

## Step 6: Data Cleaning & Standardization (Optional)

If you want to clean up your data—remove null-only columns, handle duplicates, standardize column names—run this bad boy.

In [ ]:
if all_records:
    # Create a copy for cleaning
    df_clean = df.copy()

    print("🧹 Data Cleaning Options\n")

    # Option 1: Remove columns that are entirely null
    null_columns = df_clean.columns[df_clean.isnull().all()]
    if len(null_columns) > 0:
        print(f"   Removing {len(null_columns)} completely empty columns:")
        for col in null_columns:
            print(f"      - {col}")
        df_clean = df_clean.drop(columns=null_columns)

    # Option 2: Standardize column names
    print(f"\n   Standardizing column names (lowercase, no special chars)")
    df_clean = standardize_columns(df_clean)

    # Option 3: Remove duplicate rows (optional)
    duplicates_before = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    duplicates_removed = duplicates_before - len(df_clean)
    if duplicates_removed > 0:
        print(f"\n   Removed {duplicates_removed} duplicate rows")

    print(f"\n✓ Cleaned dataframe: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")
    print(f"\n📌 Updated column names:")
    for i, col in enumerate(df_clean.columns, 1):
        print(f"   {i:>2}. {col}")

## Step 7: Preview Your Data Before Export

Take a look at what you're about to download. Make sure everything looks right.

In [ ]:
# Use the cleaned version if it exists, otherwise use the original
final_df = df_clean if 'df_clean' in locals() else df

print("📊 Final Data Preview\n")
print(f"Shape: {final_df.shape[0]} rows × {final_df.shape[1]} columns\n")
print("First 10 rows:")
print(final_df.head(10))
print("\n... [middle data omitted] ...\n")
print("Last 5 rows:")
print(final_df.tail(5))

## Step 8: Export to CSV

Here we go. This is the moment of truth. Your JSON data, now a beautiful CSV, ready to download.

In [ ]:
if all_records:
    # Create filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f"combined_data_{timestamp}.csv"

    # Export to CSV
    final_df.to_csv(csv_filename, index=False, encoding='utf-8')

    # Get file size
    file_size = Path(csv_filename).stat().st_size
    file_size_mb = file_size / (1024 * 1024)

    print(f"✅ CSV EXPORTED SUCCESSFULLY\n")
    print(f"   Filename: {csv_filename}")
    print(f"   Size: {file_size:,} bytes ({file_size_mb:.2f} MB)")
    print(f"   Rows: {final_df.shape[0]:,}")
    print(f"   Columns: {final_df.shape[1]}")
    print(f"\n🚀 Downloading now...")

    # Download the file
    files.download(csv_filename)

    print(f"\n✅ Download complete. Check your downloads folder.")
else:
    print("❌ No data to export.")

## Step 9: Advanced Options (Optional)

Want to do something fancier? Here are some advanced options you can run if you're feeling adventurous.

### Option A: Export Multiple Formats

CSV boring you? Export as Excel, JSON, or even Parquet.

In [ ]:
if all_records:
    final_df = df_clean if 'df_clean' in locals() else df
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Option 1: Excel
    try:
        excel_filename = f"combined_data_{timestamp}.xlsx"
        final_df.to_excel(excel_filename, index=False)
        print(f"✓ Exported to Excel: {excel_filename}")
        files.download(excel_filename)
    except Exception as e:
        print(f"⚠️  Excel export failed: {e}")

    # Option 2: JSON (flattened)
    try:
        json_filename = f"combined_data_{timestamp}.json"
        final_df.to_json(json_filename, orient='records', indent=2)
        print(f"✓ Exported to JSON: {json_filename}")
        files.download(json_filename)
    except Exception as e:
        print(f"⚠️  JSON export failed: {e}")

### Option B: Data Statistics & Analysis

Want a quick statistical breakdown of your data?

In [ ]:
if all_records:
    final_df = df_clean if 'df_clean' in locals() else df

    print("📈 Statistical Summary\n")
    print(final_df.describe())

    print("\n📊 Data Info:")
    final_df.info()

    print("\n🔗 Correlation Matrix (numeric columns only):")
    numeric_df = final_df.select_dtypes(include=[np.number])
    if not numeric_df.empty:
        print(numeric_df.corr())
    else:
        print("No numeric columns found.")

### Option C: Filter & Export Specific Columns

Only want certain columns in your CSV? Filter and re-export.

In [ ]:
if all_records:
    final_df = df_clean if 'df_clean' in locals() else df

    print("📌 All available columns:")
    for i, col in enumerate(final_df.columns, 1):
        print(f"   {i:>2}. {col}")

    print("\n💡 To filter columns, uncomment and modify the lines below:")
    print("\n# Example: Keep only these columns")
    print("# columns_to_keep = ['column1', 'column2', 'column3']")
    print("# df_filtered = final_df[columns_to_keep]")
    print("# timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')")
    print("# df_filtered.to_csv(f'filtered_data_{timestamp}.csv', index=False)")
    print("# files.download(f'filtered_data_{timestamp}.csv')")

    # If you want to actually use this, uncomment below and modify:
    # columns_to_keep = ['column1', 'column2', 'column3']
    # df_filtered = final_df[columns_to_keep]
    # timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    # df_filtered.to_csv(f'filtered_data_{timestamp}.csv', index=False)
    # files.download(f'filtered_data_{timestamp}.csv')

## Final Notes

### What You Got

- **A robust JSON parser** that handles nested structures like a champ
- **Automatic flattening** of complex data hierarchies
- **Error handling** that doesn't crash and burn when things go wrong
- **Multiple export formats** if you get bored with CSV
- **Data cleaning options** to make your data actually usable
- **Detailed logging** so you know exactly what happened to every single file

### Pro Tips

1. **Large datasets?** This notebook can handle thousands of files and millions of records. Go wild.
2. **Nested JSON?** The flattening function creates keys like `parent_child_grandchild`, so nothing gets lost.
3. **Want to modify behavior?** All the heavy lifting is in those helper functions. Tweak them as needed.
4. **Time tracking?** The CSV filename includes a timestamp, so you never accidentally overwrite anything.

### Troubleshooting

**Problem:** "Invalid JSON error"
**Solution:** Make sure your JSON files are actually valid JSON. Use a linter before uploading.

**Problem:** "Memory error with huge datasets"
**Solution:** Process files in batches. Upload 500 files at a time instead of all 5000.

**Problem:** "Weird column names after export"
**Solution:** Run Step 6 to standardize column names. It'll clean everything up.

---

**That's it. You now have a production-grade JSON-to-CSV converter. Use it wisely.**